# MH-01. MH PheCode nodes + cohort + spine skeleton

Broad mental-disorders cohort (NOT psychosis-risk). Locked decisions:

- Cohort = persons with at least one condition mapping to an MH-category PheCode (v1.2),
  AND whose earliest MH dx (index) falls at age 18-35 inclusive. Earlier-than-18 index
  is excluded (index is NOT reset to a later dx).
- index_date = min(condition_start_date) over MH dx. Tie-break: smallest 3-digit parent,
  then smallest full PheCode (numeric).
- ICD-9-CM / ICD-10-CM -> PheCode only. SNOMED-only dx are counted as unmapped (not mapped).
- years_in_ehr_before_index derives from observation_period (single source, reused in MH-04).

Scope cuts (agreed): no FAMD/MFA, no ADI/SVI, no note_nlp, no SNOMED mapping.

Output: `output/mh/spine.parquet` + node list + unmapped counts.


## 1. Config and external file checks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2836994-data2'      # same OMOP release as psychosis pipeline
PHECODE_MAP_PATH = 'phecode_icd10.csv'         # ICD -> PheCode crosswalk (confirm ICD-9 + ICD-10)
PHECODE_DEF_PATH = 'phecode_definitions1.2.csv'  # PheCode -> description + category (confirm exists)
OUTPUT_DIR = Path('output/mh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Locked cohort parameters
AGE_MIN, AGE_MAX = 18, 35          # inclusive; earlier-than-min index is excluded, not reset
CROSSWALK_VERSION = 'phecode_v1.2' # recorded in provenance

# MH 3-digit parent categories (v1.2 mental disorders)
MH_PARENTS = {295, 296, 300, 301, 303, 304, 305, 306, 313, 316, 317, 318}
PSYCHOSIS_PARENT = 295             # 295.x force-retained downstream; later-psychosis outcome

con = duckdb.connect()
print('DuckDB:', duckdb.__version__)
for p in [PHECODE_MAP_PATH, PHECODE_DEF_PATH]:
    print(f'{p:40s} exists = {Path(p).exists()}')
print('DATA_PATH:', DATA_PATH)


## 2. Build MH PheCode node set from the crosswalk

In [ ]:
# ICD -> PheCode map. Expect columns like: icd, phecode (+ maybe flag for ICD-9/10).
phe_map = pd.read_csv(PHECODE_MAP_PATH, dtype=str)
phe_map.columns = [c.strip().lower() for c in phe_map.columns]
print('crosswalk columns:', list(phe_map.columns))

ICD_COL = 'icd' if 'icd' in phe_map.columns else phe_map.columns[0]
PHE_COL = 'phecode' if 'phecode' in phe_map.columns else phe_map.columns[1]
phe_map = phe_map[[ICD_COL, PHE_COL]].dropna().rename(columns={ICD_COL:'icd', PHE_COL:'phecode'})
phe_map['icd'] = phe_map['icd'].str.strip().str.upper()
phe_map['phecode'] = phe_map['phecode'].str.strip()

def parent3(phecode):
    try:
        return int(float(phecode))     # 3-digit integer parent, e.g. 296.2 -> 296
    except Exception:
        return np.nan

phe_map['parent3'] = phe_map['phecode'].map(parent3)
mh_map = phe_map[phe_map['parent3'].isin(MH_PARENTS)].copy()
mh_nodes = sorted(mh_map['phecode'].unique(), key=lambda s: float(s))
print(f'MH ICD->PheCode rows: {len(mh_map):,} | distinct MH nodes: {len(mh_nodes)}')

# PheCode labels + category (for index_concept_label and MH category names)
if Path(PHECODE_DEF_PATH).exists():
    defs = pd.read_csv(PHECODE_DEF_PATH, dtype=str)
    defs.columns = [c.strip().lower() for c in defs.columns]
    label_col = next((c for c in ['phenotype','description','phecode_str'] if c in defs.columns), None)
    node_labels = (defs[['phecode', label_col]].rename(columns={label_col:'label'})
                   if label_col else pd.DataFrame(columns=['phecode','label']))
else:
    print('WARNING: PHECODE_DEF_PATH missing -> labels will be blank, category from parent3 only')
    node_labels = pd.DataFrame({'phecode': mh_nodes, 'label': ['']*len(mh_nodes)})

node_table = (pd.DataFrame({'phecode': mh_nodes})
              .assign(parent3=lambda d: d['phecode'].map(parent3))
              .merge(node_labels, on='phecode', how='left'))
node_table['is_psychosis'] = node_table['parent3'] == PSYCHOSIS_PARENT
node_table.to_parquet(OUTPUT_DIR / 'mh_node_table.parquet', index=False)
print('Saved node table:', node_table.shape)
node_table.head()


## 3. Map each patient's diagnoses to MH PheCodes (ICD-9/10 only)

In [ ]:
# Register the crosswalk so DuckDB can join condition rows to PheCodes.
con.register('mh_map', mh_map[['icd','phecode','parent3']])

def gp(table):
    return f"read_parquet('{DATA_PATH}/{table}/*.parquet')"

# condition_source_value carries the raw ICD string in this Epic/OMOP release.
mh_dx = con.sql(f'''
SELECT c.person_id,
       c.condition_start_date AS dx_date,
       m.phecode, m.parent3
FROM {gp('condition_occurrence')} c
JOIN mh_map m
  ON UPPER(TRIM(c.condition_source_value)) = m.icd
WHERE c.condition_start_date IS NOT NULL
''').to_df()
mh_dx['dx_date'] = pd.to_datetime(mh_dx['dx_date'])
print('MH-mapped dx rows:', f'{len(mh_dx):,}', '| persons:', f"{mh_dx['person_id'].nunique():,}")

# Unmapped / source-only accounting: MH-looking source values that did NOT map.
# (Coarse: rows whose source value never appears in the crosswalk at all.)
unmapped = con.sql(f'''
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT c.person_id) AS n_persons
FROM {gp('condition_occurrence')} c
LEFT JOIN mh_map m ON UPPER(TRIM(c.condition_source_value)) = m.icd
WHERE m.icd IS NULL
''').to_df()
print('Source values not in crosswalk (incl. SNOMED-only):')
print(unmapped)


## 4. Index date, tie-break, and 18-35 age gate

In [ ]:
# Attach birth_datetime for age_at_index.
persons = con.sql(f"SELECT person_id, birth_datetime, gender_source_value FROM {gp('person')}").to_df()
persons['birth_datetime'] = pd.to_datetime(persons['birth_datetime'])

# Earliest MH dx date per person = candidate index.
idx_date = mh_dx.groupby('person_id', as_index=False)['dx_date'].min().rename(columns={'dx_date':'index_date'})

# Tie-break among dx on the index date: smallest parent3, then smallest full phecode.
tie = mh_dx.merge(idx_date, on='person_id')
tie = tie[tie['dx_date'] == tie['index_date']].copy()
tie['phe_num'] = tie['phecode'].astype(float)
tie = tie.sort_values(['person_id','parent3','phe_num'])
index_pick = tie.groupby('person_id', as_index=False).first()[['person_id','index_date','parent3','phecode']]
index_pick = index_pick.rename(columns={'parent3':'index_signal_type','phecode':'index_phecode'})

spine = index_pick.merge(persons, on='person_id', how='left')
spine['age_at_index'] = ((spine['index_date'] - spine['birth_datetime']).dt.days // 365.25).astype('Int64')

before = len(spine)
spine = spine[(spine['age_at_index'] >= AGE_MIN) & (spine['age_at_index'] <= AGE_MAX)].copy()
print(f'Persons with MH dx: {before:,} -> after 18-35 index gate: {len(spine):,}')

# Assertions: gate is exactly inclusive and index is the earliest MH dx.
assert spine['age_at_index'].between(AGE_MIN, AGE_MAX).all()
assert (spine.merge(idx_date, on='person_id')['index_date_x']
        .eq(spine.merge(idx_date, on='person_id')['index_date_y']).all())
print('age gate + index assertions passed')


## 5. Spine fields from observation_period and visits

In [ ]:
# years_in_ehr_before_index + post_index_observation_days from observation_period.
# This observation_period-based field is the SOLE lookback filter in MH-04.
op = con.sql(f'''
SELECT person_id,
       MIN(observation_period_start_date) AS op_start,
       MAX(observation_period_end_date)   AS op_end
FROM {gp('observation_period')}
GROUP BY person_id
''').to_df()
op['op_start'] = pd.to_datetime(op['op_start'])
op['op_end'] = pd.to_datetime(op['op_end'])

spine = spine.merge(op, on='person_id', how='left')
spine['years_in_ehr_before_index'] = (spine['index_date'] - spine['op_start']).dt.days / 365.25
spine['post_index_observation_days'] = (spine['op_end'] - spine['index_date']).dt.days
spine['years_in_ehr_before_index'] = spine['years_in_ehr_before_index'].clip(lower=0)
spine['post_index_observation_days'] = spine['post_index_observation_days'].clip(lower=0)

# index_concept_label from the node table.
spine = spine.merge(node_table[['phecode','label']].rename(columns={'phecode':'index_phecode',
                    'label':'index_concept_label'}), on='index_phecode', how='left')
spine['sex'] = spine['gender_source_value']

# all_negative / all_masked are filled in MH-02 (need Y); placeholders here.
spine['all_negative'] = pd.NA
spine['all_masked'] = pd.NA

spine = spine.sort_values('person_id').reset_index(drop=True)
spine_cols = ['person_id','index_date','age_at_index','index_signal_type','index_phecode',
              'index_concept_label','sex','years_in_ehr_before_index',
              'post_index_observation_days','all_negative','all_masked']
spine = spine[spine_cols]
spine.to_parquet(OUTPUT_DIR / 'spine.parquet', index=False)
mh_dx.sort_values('person_id').to_parquet(OUTPUT_DIR / 'mh_dx_events.parquet', index=False)

print('spine:', spine.shape)
print('index_signal_type distribution:')
print(spine['index_signal_type'].value_counts().sort_index())
spine.head()


## 6. Provenance stub

In [ ]:
prov = {
    'crosswalk_version': CROSSWALK_VERSION,
    'age_gate': [AGE_MIN, AGE_MAX],
    'n_mh_nodes': len(mh_nodes),
    'n_cohort': int(len(spine)),
    'unmapped_rows': int(unmapped['n_rows'].iloc[0]),
    'unmapped_persons': int(unmapped['n_persons'].iloc[0]),
    'scope_cuts': ['no_FAMD','no_ADI_SVI','no_note_nlp','no_SNOMED_mapping'],
}
import json
(OUTPUT_DIR / 'provenance_mh01.json').write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))
